# 20 — Model-Aware Prompt Engineering

    ## Scenario and success criteria

    Northstar compares model profiles behind one typed contract rather than scattering provider-specific behavior across the application.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Separate durable contracts from provider adapters.
- Compare quality, schema validity, latency, and cost.
- Choose by measured constraints, not model branding.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 20 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A model migration can change output, tool use, latency, safety behavior, and token accounting.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab20 import Entity, ModelProfile, choose_profile, run_trial

text = "Google was founded in 1998 as a search company."
expected = Entity(company="Google", year=1998, industry="search")
profiles = [
    ModelProfile("small", True, 32000, 70, 3),
    ModelProfile("large", True, 1000000, 240, 12),
]

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
trials = [run_trial(profile, text, expected) for profile in profiles]
for trial in trials:
    print(trial)
print("selected", choose_profile(trials, latency_budget_ms=100, cost_budget=5))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert all(trial.correct and trial.contract_valid for trial in trials)
assert choose_profile(trials, latency_budget_ms=100, cost_budget=5) == "small"
assert choose_profile(trials, latency_budget_ms=50, cost_budget=5) is None

## Production upgrade

Maintain provider conformance tests, pin known-good configurations, observe model and adapter versions, test migrations on fixed slices, and preserve a rollback path. Capabilities and prices are time-sensitive operational data.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.